## Regressione Lineare

![least_square](images/least_square_reg.png)

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.fx.operator_schemas import ArgsKwargsPair

dfm = pd.read_csv("data/mpg.csv").dropna()
print(dfm.info())
dfm.head()

Vogliamo imparare un **modello lineare semplice** che che ci parmetta di predire il consumo `mpg` di una macchina a partire dalla sua potenza `hp`.

Dobbiamo trovare i migliori $\alpha$ e $\beta$ per cui $\hat{y} = \alpha \cdot x + \beta$

con x=`hp`(horsepower) y=`mpg` (miles-per-gallon)

Per farlo, useremo il metodo della **discesa del gradiente**:

![sgd](images/sgd.png) 

In [ ]:
hp = torch.tensor(dfm["horsepower"].to_numpy(), dtype=torch.float32)
x = (hp - hp.mean()) / hp.std()

y  = torch.tensor(dfm["mpg"].to_numpy(), dtype=torch.float32)

**Nota**: Perché standardizziamo `hp`?

La discesa del gradiente lavora meglio se la feature è centrata e su scala ≈1. Con valori molto grandi (come la potenza), il “paesaggio” della loss è una valle lunga e stretta: la discesa rimbalza tra le pareti e serve un passo piccolissimo ogni volta.

Standardizzando, ogni valore diventa quante deviazioni standard dista dalla media, la "valle" diventa una conca rotonda e la discesa va dritta al fondo, più veloce e stabile.

Non cambia il significato dei dati: solo le “coordinate” in cui ottimizziamo.

In [ ]:
# definiamo con torch i parametri da ottimizzare con discesa del gradiente
alpha = torch.tensor([0.0], requires_grad=True)
beta = torch.tensor([0.0], requires_grad=True)
lr = 0.1  # definiamo la learning rate
# ottimizzatore
opt = torch.optim.SGD([alpha, beta], lr=lr)

In `pytorch`:
 1. `backward()` calcola le pendenze
 2. `step()` fa il passo
 3. `zero_grad()` azzera prima del successivo.

**NB**: Questo va ripetuto ad ogni epoca di training, indipendentemente dalla funzione di loss / costo usata!

In [ ]:
n_epoche = 300
for epoca in range(n_epoche):
    y_pred = alpha * x + beta
    loss = ((y_pred - y)**2).mean() # MSE
    opt.zero_grad() # 1) azzera i gradienti
    loss.backward() # 2) calcola i gradienti
    opt.step() # 3) fai il passo in

Vediamo il risultato del training: cosa possiamo osservare?

In [ ]:
ordine = hp.argsort()
plt.scatter(hp, y, s=12, alpha=.6)
plt.plot(hp[ordine], (alpha * x + beta).detach()[ordine], color="red", lw=3)
plt.xlabel("potenza (hp)"); plt.ylabel("consumo (mpg)");

Forse il modello lineare semplice non è adatto a descrivere la relazione tra $x$ e $y$, quindi potrebbe non fornirci previsioni adeguate su dati di automobili che non conosciamo.

Aggiungiamo una feature, per vedere se otteniamo una **rappresentazione migliore dei dati**, ovvero una che spieghi meglio la relazione tra potenza e consumo in questo dataset.

Come nuova feature, aggiungiamo il quadrato di `hp`, `hp`$^2$, e cerchiamo tramite ottimizzazione del MSE con discesa del gradiente i parametri di:

$\hat{y} = \alpha_1 x + \alpha_2 x^2 + \beta$

dove $x=$`hp` $x^2=$`hp`$^2$ $y=$`mpg`

Mettendo $x$ e $x^2$ come colonne di una matrice $X = [x, x^2]$, e coerentemente i coefficienti angolari $\alpha = [\alpha_1, \alpha_2]$ recuperiamo la notazione di prima, ma con dimensioni diverse.


In [ ]:
X = torch.stack([x, x**2], dim=1) # due feature per ogni auto
Alpha = torch.zeros(2, requires_grad=True)
beta = torch.zeros(1, requires_grad=True)
lr = 0.1
n_epoche = 300
opt = torch.optim.SGD([Alpha, beta], lr=lr)
for epoca in range(n_epoche):
    y_pred = X @ Alpha + beta
    loss = ((y_pred - y)**2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

Adesso non abbiamo più una funzione che varia linearmente con `hp`: cosa possiamo osservare?

In [ ]:
ordine = hp.argsort()
plt.scatter(hp, y, s=12, alpha=.6)
plt.plot(hp[ordine], (X @ Alpha + beta).detach()[ordine], color="green", lw=3)
plt.xlabel("potenza (hp)");
plt.ylabel("consumo (mpg)");
plt.show()

Adesso abbiamo provato a indovinare, aggiungendo a mano una feature, quale fosse la rappresentazione migliore dei dati per il nostro problema di capire cosa lega la potenza al consumo:

**A partire da un dataset e da una funzione di costo / perdita da ottimzzare, le reti neurali imparano automaticamente la rappresentazione migliore dei dati per minimizzarla.**

## E se ho un problema di classificazione? Regressione Logistica!

A dispetto del nome, è il gemello della regressione lineare per la classificazione: stesso loop, cambia l'ultimo passo:

![simgoid](images/sigmoid.png)

In [ ]:
dfp = pd.read_csv("data/penguins.csv").dropna()
dfp.info()

Il nostro obiettivo è distinguere i pinguini di specie `Gentoo` dal resto, utilizzando come features la lunghezza delle pinne `flipper_length_mm` e la lunghezza del becco `beak_length_mm`.

In [ ]:
feat = dfp[["beak_length_mm", "flipper_length_mm"]].to_numpy()
feat = (feat - feat.mean(0)) / feat.std(0) # standardizziamo le features (v. sopra perchè)
X = torch.tensor(feat, dtype=torch.float32)               # (N, 2)
y = torch.tensor((dfp["species"] == "Gentoo").to_numpy(), dtype=torch.float32)  # 1 = Gentoo, 0 = altri

In [ ]:
# stessi elementi (coefficiente, intercetta e ottimizzatore) usati nella regressione lineare
Alpha = torch.zeros(2, requires_grad=True)
beta = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([Alpha, beta], lr=0.1)

Nella classificazione, prima di calcolare l'errore e ottimizzare la perdita con discesa del gradiente, si assegna ogni punto ad una classe (una specie nel nostro caso).

Questo implica che anche la funzione di perdita non potrà essere lo scarto quadratico medio (che funziona solo per la regressione) ma sarà la **Binary Cross Entropy Loss**, che come suggerisce il nome è indicata per la classificazione binaria.

In [ ]:
n_epoche = 300
for epoca in range(n_epoche):
    z = X @ Alpha + beta  # combinazione lineare (come la regressione)
    p = torch.sigmoid(z) # la sigmoide schiaccia z tra 0 e 1: una probabilità
    loss = torch.nn.functional.binary_cross_entropy(p, y)  # cross-entropy binaria
    # gli "obbligatori" nell'ottimizzazione con discesa del gradiente
    opt.zero_grad()
    loss.backward()
    opt.step()

In [ ]:
# accuratezza: quante previsioni sono corrette (soglia a 0.5)
with torch.no_grad():
    pred = (torch.sigmoid(X @ Alpha + beta) > 0.5).float()
    acc = (pred == y).float().mean()
print(f"Accuratezza: {acc.item():.1%}")